1. Imports e sementes

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch


# Garante que a pasta raiz seja encontrada mesmo rodando de dentro de notebooks/
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.train import set_seed, prepare_dataset, inverse_target

RANDOM_STATE = 42
SEEDS = [42, 100, 202, 303, 404]  # 5 sementes para rodar os experimentos depois
BATCH_SIZE = 256

set_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Rodando em:", device)

Rodando em: cpu


2. Carregar e conferir o dataset de regressão

In [2]:
reg_data = prepare_dataset(
    csv_path=str(ROOT_DIR / "data" / "processed" / "dataset_regression.csv"),
    task="regression",
    target_col="TARGET",
    random_state=RANDOM_STATE,
    batch_size=BATCH_SIZE,
)

for nome in ["train", "val", "test"]:
    X_part = reg_data["scaled_splits"][f"X_{nome}"]
    y_part = reg_data["scaled_splits"][f"y_{nome}"]
    print(f"{nome:5s} -> X: {X_part.shape}, y: {y_part.shape}")

xb, yb = next(iter(reg_data["loaders"]["train"]))
print("Um batch de treino -> xb:", xb.shape, xb.dtype, "| yb:", yb.shape, yb.dtype)

train -> X: (160000, 3), y: (160000,)
val   -> X: (40000, 3), y: (40000,)
test  -> X: (50000, 3), y: (50000,)
Um batch de treino -> xb: torch.Size([256, 3]) torch.float32 | yb: torch.Size([256, 1]) torch.float32


3. Testando MLP

In [3]:
from src.models import MLP

modelo_teste = MLP(
    in_features=xb.shape[1],   # 3 na regressão
    out_features=1,
    hidden_units=(64, 32),
    activation="ReLU",
    dropout=0.0,
).to(device)

with torch.no_grad():
    saida_teste = modelo_teste(xb.to(device))

n_params = sum(p.numel() for p in modelo_teste.parameters() if p.requires_grad)
print("Saída do modelo de teste:", saida_teste.shape, saida_teste.dtype)
print("Total de parâmetros treináveis:", n_params)

Saída do modelo de teste: torch.Size([256, 1]) torch.float32
Total de parâmetros treináveis: 2369


4. Treinamento Baseline (MLP com MSELoss e Early Stopping)

In [4]:
import importlib
import src.train
importlib.reload(src.train)

from src.train import set_seed, treinar_modelo

SEMENTE_BASE = SEEDS[0]  # 42
set_seed(SEMENTE_BASE)

modelo_base = MLP(
    in_features=reg_data["n_features"],
    out_features=1,
    hidden_units=(64, 32),
    activation="ReLU",
    dropout=0.0,
).to(device)

loss_fn = torch.nn.MSELoss()
otimizador = torch.optim.Adam(modelo_base.parameters(), lr=1e-3)

resultado = treinar_modelo(
    modelo=modelo_base,
    dl_treino=reg_data["loaders"]["train"],
    dl_val=reg_data["loaders"]["val"],
    loss_fn=loss_fn,
    otimizador=otimizador,
    device=device,
    n_epocas=100,
    paciencia=15,
)

print(f"Épocas rodadas: {resultado['epocas_rodadas']} | Melhor época: {resultado['melhor_epoca']}")
print(f"Melhor perda Val (MSE padronizado): {resultado['melhor_val']:.6f}")
print(f"Tempo total: {resultado['tempo_s']:.2f}s")

Épocas rodadas: 31 | Melhor época: 16
Melhor perda Val (MSE padronizado): 0.001200
Tempo total: 47.98s
